## BNN MNIST: Hardware AXI-Stream Verification (Varying Batch Sizes)

This notebook verifies the FPGA hardware for varying stream sizes (batch sizes) by comparing against the fast software golden model. This approach validates the DMA/AXI-Stream data handling, including the implicit `tlast` assertion.

**Goals:**
1. Run inference for different batch sizes (1, 2, random, large).
2. Use the fast `bnn_sw.inference_batch` function for golden model reference.
3. Check for bit-exact match (HW output == SW output).

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time
import random

from bnn_mnist import BNN_MNIST

In [ ]:
ol = Overlay('design_1.bit')
dma = ol.axi_dma_0
dma_send = dma.sendchannel
dma_recv = dma.recvchannel
bnn_sw = BNN_MNIST()
print("Hardware Overlay and Software Model loaded successfully.")

In [ ]:
mnist = np.load("dataset/mnist_test_data_original.npy", allow_pickle=True)
X_global = np.reshape(mnist.item().get("data"), (10000, 784))
print("Test dataset loaded.")

### Helper Function for Packing Input Data

In [ ]:
def pack_batch(images, bnn_sw):
    """Packs a batch of images into the 25x uint32 format required by the FPGA."""
    num = len(images)
    # INPUT_PACKED_WIDTH = 800 bits / 32 bits per word = 25
    INPUT_PACKED_WIDTH = 25
    
    # Allocate contiguous memory buffer for Pynq DMA
    packed_buffer = allocate(shape=(num, INPUT_PACKED_WIDTH), dtype=np.uint32)
    
    for i in range(num):
        # 1. Binarize input to bipolar {-1, 1}
        img_bi = bnn_sw.sign(bnn_sw.adj(images[i]))
        
        # 2. Pad to 800 bits (add 16 ones) -> 784 + 16 = 800
        # NOTE: Using '1' (bipolar) for padding means '0' (quantized/packed) bit
        img_padded = np.append(img_bi, [1] * 16)
        
        # 3. Pack into 32-bit words
        # bnn_sw.pack converts {-1, 1} to {0, 1} (quantized) and packs
        packed_buffer[i] = bnn_sw.pack(img_padded, 800)
        
    return packed_buffer

In [ ]:
def run_verification_test(num_samples):
    if num_samples <= 0 or num_samples > 10000: return True
    
    # Select samples (sequential indices)
    indices = np.arange(num_samples)
    X_batch = X_global[indices]
    
    print(f"\n>>> Testing Stream Size: {num_samples}")
    print("--------------------------------------------------")
    
    # 1. Run Golden Model (Software)
    start_sw = time.time()
    # We use the fast batch inference function
    sw_predictions = bnn_sw.inference_batch(X_batch)
    time_sw = time.time() - start_sw
    
    # 2. Prepare and Run Hardware
    in_buffer = pack_batch(X_batch, bnn_sw)
    # Output is a single prediction (int32) per sample
    out_buffer = allocate(shape=(num_samples,), dtype=np.int32)
    
    start_hw = time.time()
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()
    time_hw = time.time() - start_hw
    
    # 3. Compare Results
    # The hardware output array (out_buffer) contains the predicted labels.
    hw_predictions = out_buffer.astype(np.int32)
    
    # Compare all results simultaneously
    matches = np.sum(hw_predictions == sw_predictions)
    mismatches = num_samples - matches
    
    print(f"SW Inference Time: {time_sw*1000:.2f} ms")
    print(f"HW Inference Time: {time_hw*1000:.2f} ms")
    print(f"Total Samples Tested: {num_samples}")
    print(f"Matches (HW == SW): {matches}")
    print(f"Mismatches: {mismatches}")
    
    # Free buffers to prevent memory leaks
    in_buffer.freebuffer()
    out_buffer.freebuffer()
    
    if mismatches == 0:
        print(">> PASSED: 100% Match with Golden Model (AXI-Stream Integrity OK).")
        return True
    else:
        print(">> FAILED: Logic or Stream Mismatch Detected.")
        # Optionally print the first mismatch for debugging
        first_mismatch_idx = np.where(hw_predictions != sw_predictions)[0][0]
        print(f"  First mismatch at input index {first_mismatch_idx}: HW={hw_predictions[first_mismatch_idx]}, SW={sw_predictions[first_mismatch_idx]}")
        return False

In [ ]:
# --- Run the Test Configurations (Similar to C++ Testbench) ---

test_sizes = [1, 2, 8, 32]

# Add 4 random sizes between 3 and 1000
random.seed(42) # Ensure reproducible random sizes
for _ in range(4):
    test_sizes.append(random.randint(3, 1000))

# Add large sizes
test_sizes.extend([1000, 1024, 2048])

total_errors = 0

for size in sorted(list(set(test_sizes))): # Sort and remove duplicates
    if not run_verification_test(size):
        total_errors += 1

print("\n*******************************************")
if total_errors == 0:
    print("SUCCESS: All variable streaming configurations passed!")
else:
    print(f"FAILURE: {total_errors} configurations resulted in mismatches.")
print("*******************************************")